# GridLeak AI — Ön İşleme ve Özellikler

Mevcut master defterin hücreleri içerikleri ve kayıtlı çıktıları korunarak bu deftere taşınmıştır. Ortak hazırlık hücreleri bağımsız açılış için tekrar yer alır.

Bu düzenleme yeni eğitim veya değerlendirme çalıştırmamıştır. Hücreleri çalıştırmak ilgili hesaplamaları yeniden yürütür.

## Ham veriden yeniden üretim (tamamlanan hazırlık akışı)
`ai/features/prepare_data.py`: kimlik/etiket kontrolü, kayıtlı bölmenin korunması
(yoksa seed=42 ile tabakalı %80/%20 ayrım), eksik günün NaN eklenmesi,
ilk 7 ve genişletilmiş 49 özellik, kayıt kalitesi ve hiç kaydı olmayanların ayrılması.
%80 doluluk yalnız dönemsel değişim hesabının şartıdır; abone eleme eşiği değildir.

Çıktılar yeni bir klasöre yazılır. Mevcut v2/v3 tablolarının üzerine yazılmaz.
Test hazırlığı yalnız teslim akışını sıfırdan üretmek için ayrıca `--include-test`
ile istenir; testte model/özellik/eşik seçimi yapılmaz.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'ai/features/prepare_data.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ai.features.prepare_data import prepare
PREPARE_NEW_DATA = False  # Hazırlığı yeniden çalıştırmak isterseniz True.
if PREPARE_NEW_DATA:
    from datetime import datetime
    output = ROOT/'data/processed'/('rebuild_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    print(prepare(ROOT/'data/raw/data.csv', output, ROOT/'data/splits/abone_bolunmesi.csv'))
else:
    print('Hazırlık fonksiyonu hazır; mevcut veriler değiştirilmedi.')


# GridLeak AI (v3) — Uçtan Uca Açıklamalı Master Defter

Bu defter, projedeki gerçek üretim modüllerini (**birebir aynı kodları**) doğrudan içe aktararak tüm makine öğrenmesi akışını adım adım çalıştırır ve görselleştirir:

- **Özellik Mühendisliği:** `ai/features/v3_features.py` (581 aday → 160 seçilmiş özellik)
- **Model Mimarisi ve Eğitim:** `ai/training/v3_final_train_and_test.py` (LightGBM + CatBoost + XGBoost + RankGauss MLP + Sigmoid Stacking)
- **Değerlendirme, TreeSHAP ve Katalog Aktarımı:** `ai/evaluation/v3_export_and_shap.py` & `export_application.py`

> **Veri Seti Bağlantısı (SGCC):** https://www.kaggle.com/datasets/bensalem14/sgcc-dataset

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
warnings.filterwarnings('ignore')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ai/features/v3_features.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Projedeki GERÇEK üretim modüllerini doğrudan içe aktarıyoruz (sıfır kod tekrarı!)
from ai.features.v3_features import load_raw_for_ids, extract_subscriber_features, FoldContextTransformer
from ai.training.v3_final_train_and_test import build_base_estimators, build_meta_estimator, ece_score

EXP = ROOT / 'ai/artifacts/iyilestirme_v3_20261123T171349Z'
V3 = ROOT / 'data/processed/v3'
lock = json.loads((EXP / 'locked_selection.json').read_text(encoding='utf-8'))
fcols = lock['features']
print('Proje Kökü:', ROOT)
print('Aktif Kilitli Model:', lock['model_name'])
print('Kilitli Özellik Sayısı:', len(fcols))

Proje Kökü: c:\Users\kadir\GridLeakAI
Aktif Kilitli Model: Calibrated Stacking v3 (LGBM + CatBoost + XGBoost + MLP)
Kilitli Özellik Sayısı: 160


## 1. Veri Seti Yapısı, %80/%20 Abone Bölünmesi ve Sınıf Dengesizliği
Aboneler kimlik (`CONS_NO`) bazında tabakalı bölünmüştür; eğitim ve test arasında **sıfır ortak abone** vardır.

In [ ]:
# Hazırlık sırasında test hedeflerini açmıyoruz.
train_df = pd.read_csv(V3 / 'egitim_ozellikleri_2016_v1.csv', dtype={'CONS_NO': str})
y_tr = train_df['FLAG'].to_numpy(int)
print('Skorlanabilir eğitim abonesi:', len(train_df))
print(train_df['FLAG'].value_counts())


## Canlı özellik çıkarımı — yalnız eğitimden 5 örnek
Kilitli bağlam dönüştürücüsüyle satır üretimi gösterilir. Bu gösterim bir çapraz doğrulama deneyi değildir.

In [ ]:
ctx_fitted = joblib.load(EXP / 'context_transformer.joblib')
v2_tr = pd.read_csv(ROOT / 'data/processed/v2/test_ozellikleri_2016_v1.csv', dtype={'CONS_NO': str})
sample_ids = v2_tr['CONS_NO'].head(5).tolist()
base49_sample = v2_tr.head(5).drop(columns=['CONS_NO', 'FLAG']).set_index(pd.Index(sample_ids, name='CONS_NO'))

arr_s, dates_s = load_raw_for_ids(sample_ids)
sub_s = extract_subscriber_features(arr_s, dates_s, base49_sample, include_history=True)
full_s = pd.concat([sub_s, ctx_fitted.transform(arr_s, sub_s)], axis=1)
selected_s = full_s[fcols]
print(f'Ham matris: {arr_s.shape} (5 abone × 1.034 gün) -> Aday özellik: {full_s.shape[1]} -> Seçilmiş: {selected_s.shape[1]}')
display(selected_s.iloc[:, :8].round(3))